# Combining curated depositions across sources

ChEMBL and BindingDB draw form the same primary literature, so many compounds are present in both datasets. **Concatenating the two depositions would double-count roughly
95% of compounds**, therefore, we explicitely include a notebok that performs the dataset combination.

This notebook only performs the join and it's not specific to any target or endpoint. To combine a different set of targets (other than CYPs), or a different endpoint, or a
different pair of sources, edit `SOURCES` and `TARGETS` in the parameters cell — no other change.

How values are combined:

- each source contributes **one value per compound** (median over its records), so a source with
  many records for a compound does not outweigh one with few
- `value_combined` is the **mean of the available sources**, which for the great majority returns
  the shared value, since the sources usually agree to within rounding
- `sources_disagree` marks compounds more than `DISAGREE_LOG` apart, where that mean is meaningless
  and the row should be excluded rather than trusted

**A caveat worth recording.** The join works because every deposition here was canonicalised by the
same `openadmet.toolkit` version, so their `OPENADMET_INCHIKEY` values agree. If one source deposition is ever regenerated with a different toolkit version and the other is not, this join may fail.

## Parameters

In [5]:
ENDPOINT_LABEL = "IC50"
VALUE_NAME = "pIC50"          # the comparable, log-scale value the sources are joined on

# One entry per curated deposition. `pattern` is formatted with the target name; `value` names the
# per-record column holding the log-scale value, and `extra` any columns worth carrying through.
SOURCES = {
    "chembl": {
        "path": "../activities/ChEMBL_pChEMBL_IC50/ChEMBL37_IC50_curated",
        "pattern": "ChEMBL_IC50_CYP_{target}_*_raw.parquet",
        "value": "pchembl_value",
        "extra": ["CHEMBL_INCHIKEY"],
        "version": "ChEMBL37",
    },
    "bindingdb": {
        "path": "../activities/BindingDB_IC50/BindingDB20261019_IC50_curated",
        "pattern": "BindingDB_IC50_CYP_{target}_*_raw.parquet",
        "value": "pIC50",
        "extra": [],
        "version": "BindingDB20261019",
    },
}

TARGETS = ["CYP1A2", "CYP2B6", "CYP2C8", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]

KEY = "OPENADMET_INCHIKEY"
SMILES = "OPENADMET_CANONICAL_SMILES"
DISAGREE_LOG = 1.0            # sources further apart than this are flagged, not averaged away

LOCATION = "_".join([s["version"] for s in SOURCES.values()]) + f"_{ENDPOINT_LABEL}_combined"
BUCKET = "openadmet-data-public-dev"
PUSH_TO_S3 = False  # set True only after reviewing local output; never push unreviewed data

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

location_path = Path(LOCATION)
location_path.mkdir(exist_ok=True)

## Load and combine

`load_source` reduces one deposition to one value per compound. Everything below is source-agnostic:
it only needs the key column, a value column, and optional extras.

In [9]:
def load_source(name, spec, target):
    """One row per compound from a single curated deposition, or None when the target is absent."""
    matches = sorted(Path(spec["path"]).glob(spec["pattern"].format(target=target)))
    if not matches:
        return None
    df = pd.read_parquet(matches[0])
    agg = {
        f"{VALUE_NAME}_{name}": (spec["value"], "median"),
        f"n_{name}": (spec["value"], "count"),  # 'count' skips NaNs; 'size' includes them
        SMILES: (SMILES, "first"),
    }
    for col in spec.get("extra", []):
        if col in df.columns:
            agg[col] = (col, "first")
    return df.groupby(KEY).agg(**agg)
    
def combine_target(target):
    """Outer-join every source on compound identity, one row per compound."""
    present = {
        name: df
        for name, spec in SOURCES.items()
        if (df := load_source(name, spec, target)) is not None
    }
    if not present:
        return None

    # Consolidate SMILES cleanly without suffix collisions
    smiles_series = (
        pd.concat([df[SMILES] for df in present.values()], axis=1)
        .bfill(axis=1)
        .iloc[:, 0]
        .rename(SMILES)
    )

    # Join source-specific columns (dropping SMILES first so no suffixes are needed)
    merged = pd.concat(
        [df.drop(columns=[SMILES]) for df in present.values()],
        axis=1,
        join="outer",
    )
    merged.insert(0, SMILES, smiles_series)

    # Counts & presence flags
    value_cols = [f"{VALUE_NAME}_{n}" for n in present]
    count_cols = [f"n_{n}" for n in present]
    merged[count_cols] = merged[count_cols].fillna(0).astype(int)

    found = merged[value_cols].notna()
    merged["n_sources"] = found.sum(axis=1)

    # Vectorized source labeling (handles 1, 2, or N sources without .apply)
    source_names = list(present.keys())
    if len(source_names) == 1:
        merged["source"] = f"{source_names[0]} only"
    else:
        merged["source"] = np.where(
            merged["n_sources"] == len(source_names),
            "both" if len(source_names) == 2 else "all",
            found.dot(pd.Index(source_names)) + " only",
        )

    # Summary metrics & metadata
    merged[f"{VALUE_NAME}_combined"] = merged[value_cols].mean(axis=1)
    merged["source_spread"] = merged[value_cols].max(axis=1) - merged[value_cols].min(axis=1)
    merged["sources_disagree"] = merged["source_spread"] > DISAGREE_LOG

    for name, spec in SOURCES.items():
        merged[f"{name}_source_version"] = spec["version"]

    merged.insert(0, "target", target)
    return merged.reset_index()

## Main loop

One file per target, plus a wide table with one row per compound and a column per target — the
shape a multi-task model wants. A null there means the compound was never measured against that
target, not that it was inactive.

In [10]:
combined = {}
summaries = {}
for target in TARGETS:
    df = combine_target(target)
    if df is None:
        print(f"{target}: not present in any source, skipping")
        continue
    combined[target] = df
    summaries[target] = {
        "compounds": len(df),
        **{f"{n}_only": int((df["source"] == f"{n} only").sum()) for n in SOURCES},
        "both": int((df["source"] == "both").sum()),
        "disagree": int(df["sources_disagree"].sum()),
    }

    fname = f"combined_{ENDPOINT_LABEL}_{target}.parquet"
    df.to_parquet(location_path / fname, index=False)
    print(f"  wrote {fname} (rows={len(df)})")

summary_table = pd.DataFrame(summaries).T
summary_table.to_csv(location_path / f"combination_summary_{LOCATION}.csv")
summary_table

  wrote combined_IC50_CYP1A2.parquet (rows=1645)
  wrote combined_IC50_CYP2B6.parquet (rows=238)
  wrote combined_IC50_CYP2C8.parquet (rows=493)
  wrote combined_IC50_CYP2C9.parquet (rows=2908)
  wrote combined_IC50_CYP2C19.parquet (rows=1681)
  wrote combined_IC50_CYP2D6.parquet (rows=3082)
  wrote combined_IC50_CYP3A4.parquet (rows=5717)


,compounds,chembl_only,bindingdb_only,both,disagree
CYP1A2,1645,263,109,1273,2
CYP2B6,238,25,38,175,0
CYP2C8,493,58,40,395,0
CYP2C9,2908,360,168,2380,1
CYP2C19,1681,288,163,1230,4
CYP2D6,3082,361,162,2559,4
CYP3A4,5717,978,486,4253,12


In [12]:
# wide table: one row per compound, one value column per target
WIDE_PREFIX = f"OPENADMET_{VALUE_NAME}"

wide = None
for target, df in combined.items():
    part = df[[SMILES, KEY, f"{VALUE_NAME}_combined"]].rename(
        columns={f"{VALUE_NAME}_combined": f"{WIDE_PREFIX}_{target.lower()}"})
    if part[KEY].duplicated().any():
        raise ValueError(f"{target}: duplicate compounds, cannot pivot")
    wide = part if wide is None else wide.merge(part, on=[SMILES, KEY], how="outer")

value_cols = [f"{WIDE_PREFIX}_{t.lower()}" for t in combined]
wide.insert(2, "n_targets_measured", wide[value_cols].notna().sum(axis=1))
wide = wide.sort_values("n_targets_measured", ascending=False).reset_index(drop=True)
wide.to_parquet(location_path / f"combined_{ENDPOINT_LABEL}_all_CYPs_{VALUE_NAME}.parquet", index=False)

print(f"wide table: {len(wide)} compounds x {len(combined)} targets")
print(wide["n_targets_measured"].value_counts().sort_index().rename("compounds").to_string())
wide.head()

wide table: 9695 compounds x 7 targets
n_targets_measured
1    6610
2    1533
3     745
4     365
5     320
6      61
7      61


,OPENADMET_CANONICAL_SMILES,OPENADMET_INCHIKEY,n_targets_measured,OPENADMET_pIC50_cyp1a2,OPENADMET_pIC50_cyp2b6,OPENADMET_pIC50_cyp2c8,OPENADMET_pIC50_cyp2c9,OPENADMET_pIC50_cyp2c19,OPENADMET_pIC50_cyp2d6,OPENADMET_pIC50_cyp3a4
0,COC1=CC2=C(C=C1)NC(SCC1=CC=C(C(C)C)C=C1)=N2,OLVGVEZGCNJPGR-UHFFFAOYSA-N,7,5.829869,4.840819,4.300515,5.467987,7.398970,5.689123,5.079417
1,CCCC1=C(C2=CC=C(CN)S2)C=NC=C1,OKDXZLVZPGFBHY-UHFFFAOYSA-N,7,6.720623,7.060240,5.658789,5.070291,5.959304,5.569318,5.368266
2,NCC1=CC=C(C2=C(C3=CC=CC=C3)C=CN=C2)S1,UGCVMZPRZUHXOO-UHFFFAOYSA-N,7,6.040479,7.220924,5.959304,5.538801,5.358274,6.119593,5.277862
3,COC1=CC2=C(C=C1)NC(SCCCO)=N2,FUVYXGUCJOVMNB-UHFFFAOYSA-N,7,5.091294,5.259819,5.248976,4.300515,4.300515,5.379932,5.260214
4,CCCC1=C(C2=CC=C(CN)O2)C=NC=C1,KRLAVZKZJIVVOU-UHFFFAOYSA-N,7,6.430899,6.430899,6.582513,4.720623,5.699485,5.769776,5.309902


## Push to S3 and build the Intake catalog

In [ ]:
if PUSH_TO_S3:
    import intake
    from intake.readers.datatypes import Parquet

    from openadmet.toolkit.webservices.credentials import S3Settings
    from openadmet.toolkit.webservices.s3 import S3Bucket

    assert "AWS_ACCESS_KEY_ID" in __import__("os").environ, "set AWS credentials in the environment first"
    settings = S3Settings()
    bucket = S3Bucket.from_settings(settings, BUCKET)

    versions = {f"{n}_source_version": s["version"] for n, s in SOURCES.items()}
    cat = intake.entry.Catalog()
    for path in sorted(location_path.glob("*.parquet")):
        dest = f"{LOCATION}/{path.name}"
        bucket.push_file(path, dest)
        entry = path.stem.replace(f"combined_{ENDPOINT_LABEL}_", "")
        metadata = {
            "description": (f"{ENDPOINT_LABEL} activity for {entry}, combined across "
                            f"{', '.join(SOURCES)} and deduplicated on {KEY}."),
            "endpoint": ENDPOINT_LABEL, "value": f"{VALUE_NAME}_combined",
            "sources": list(SOURCES), **versions,
        }
        cat[entry] = intake.readers.PandasParquet(Parquet(url=bucket.to_uri(dest), metadata=metadata))

    catname = f"CATALOG_{LOCATION}.yaml"
    cat.to_yaml_file(catname)
    bucket.push_file(catname, f"{LOCATION}/{catname}")
    print(f"pushed catalog to {LOCATION}/{catname} with {len(list(cat))} entries")
else:
    print("PUSH_TO_S3 is False -- local parquet only, nothing pushed, no catalog built")